# Week 1: Foundations — Cleaning & Exploring the Applicant Dataset
**Prepared by:** Nabia Memon  
**Program:** NextGen Learners Data Science Internship

## Objective
Use Pandas to inspect, clean, standardize, validate, and export an applicant dataset. The data contains intentionally introduced issues for practice. All changes are performed in Python so the work is reproducible.

## Step 1 — Import the library
Pandas helps us load, inspect, clean, and export tabular data.

In [ ]:
import os
import pandas as pd
pd.set_option("display.max_columns", None)
print("Pandas version:", pd.__version__)

## Step 2 — Load the raw dataset
Place `applicants.csv` in the same folder as this notebook. If it is missing, this cell creates a small practice dataset so the notebook can still run.

In [ ]:
if not os.path.exists("applicants.csv"):
    sample = pd.DataFrame({
        "Applicant Name": ["Ali Khan", "Sara Memon", None, "Hina Shah", "Omar Ali"],
        "Email": ["ali@example.com", "SARA@example.com", "noor@example.com", None, "omar@example.com"],
        "Phone": ["0301-1234567", "0302 1234567", "0303-7654321", "0304 5551234", "0305-1112223"],
        "Domain Applied": ["web dev", "WEB DEV", "Data Science", "Cyber Security", "AI"],
        "University": ["University of Sindh", None, "MUET", "SZABIST", "NED University"],
        "Application Date": ["12/05/2026", "2026-05-12", "13/05/2026", "32/01/2026", "15/05/2026"],
        "Status": ["selected", "SELECTED", None, "Under Review", "rejected"]
    })
    sample.to_csv("applicants.csv", index=False)
df = pd.read_csv("applicants.csv")
print("Raw dataset loaded successfully.")
df.head()

## Step 3 — Explore before cleaning
`shape` gives the row/column count, `head()` previews records, `info()` shows types and non-null counts, `isnull().sum()` counts missing values, and `duplicated().sum()` counts exact duplicate rows.

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
display(df.head())
df.info()
print("\\nMissing values per column:")
display(df.isnull().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("\\nDomain values before cleaning:")
display(df["Domain Applied"].value_counts(dropna=False))

### Initial observations
Review the outputs above. The practice data includes missing values, duplicates, inconsistent text, different domain labels, and dates that may not parse. Counts shown are the actual counts for the loaded file.

## Step 4 — Handle missing values with a reason
- **Applicant Name:** drop rows where the name is missing because the applicant cannot be identified reliably.
- **Email:** keep rows for other analysis, but use a clearly marked placeholder for missing emails. This is not a real contact address.
- **University:** fill with `Not Specified` because the row can still be useful.
- **Status:** fill with `Under Review`, treating an unrecorded status as not yet processed.

In [ ]:
missing_name_count = int(df["Applicant Name"].isna().sum())
df = df.dropna(subset=["Applicant Name"]).copy()
print("Rows removed for missing Applicant Name:", missing_name_count)

missing_email_mask = df["Email"].isna()
missing_email_count = int(missing_email_mask.sum())
df.loc[missing_email_mask, "Email"] = [f"not_provided_{i}@example.com" for i in df.index[missing_email_mask]]
print("Missing Email values replaced with labelled placeholders:", missing_email_count)

missing_university_count = int(df["University"].isna().sum())
df["University"] = df["University"].fillna("Not Specified")
print("Missing University values filled:", missing_university_count)

missing_status_count = int(df["Status"].isna().sum())
df["Status"] = df["Status"].fillna("Under Review")
print("Missing Status values filled:", missing_status_count)

## Step 5 — Remove exact and near-duplicate applications
Remove fully identical rows, then repeated email addresses. Email is a more useful identifier than a name because different people can share a name. Placeholder emails are unique.

In [ ]:
exact_duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Exact duplicate rows removed:", exact_duplicates_before)

df["Email"] = df["Email"].astype("string").str.strip().str.lower()
duplicate_email_mask = df["Email"].duplicated(keep="first")
duplicate_email_count = int(duplicate_email_mask.sum())
df = df.loc[~duplicate_email_mask].copy()
print("Additional rows removed due to repeated Email:", duplicate_email_count)
print("Rows remaining:", len(df))

## Step 6 — Standardize text fields
Remove extra spaces, standardize names/status/university, and map domain variations to five approved labels. Any unknown labels are shown for review rather than guessed.

In [ ]:
for col in ["Applicant Name", "Email", "Phone", "Domain Applied", "University", "Status"]:
    df[col] = df[col].astype("string").str.strip()
df["Applicant Name"] = df["Applicant Name"].str.title()
df["Email"] = df["Email"].str.lower()
df["University"] = df["University"].str.title()
df["Status"] = df["Status"].str.lower().str.title()

domain_key = df["Domain Applied"].str.lower().str.replace(r"[^a-z]", "", regex=True)
domain_mapping = {
    "webdev": "Web Development", "webdevelopment": "Web Development",
    "datascience": "Data Science", "cybersecurity": "Cyber Security",
    "artificialintelligence": "Artificial Intelligence", "ai": "Artificial Intelligence",
    "appdev": "App Development", "appdevelopment": "App Development"
}
df["Domain Applied"] = domain_key.map(domain_mapping)
print("Domain values after standardization:")
print(df["Domain Applied"].value_counts(dropna=False))
print("Unique standardized domain labels:", sorted(df["Domain Applied"].dropna().unique().tolist()))
print("Unmapped/missing domain values:", int(df["Domain Applied"].isna().sum()))

In [ ]:
unmapped_domains = df.loc[df["Domain Applied"].isna()]
if len(unmapped_domains):
    print("Rows needing domain review:")
    display(unmapped_domains[["Applicant Name", "Email", "Domain Applied"]])
else:
    print("All domain labels were mapped to the approved categories.")

## Step 7 — Fix data types and phone formatting
`pd.to_datetime(..., errors="coerce")` converts readable dates to datetime and changes invalid dates to `NaT` rather than crashing. Rows with invalid/missing dates are removed because the date is needed for date-based analysis. Phone numbers stay as text to preserve leading zeros.

In [ ]:
df["Application Date"] = pd.to_datetime(df["Application Date"], errors="coerce", dayfirst=True)
invalid_date_count = int(df["Application Date"].isna().sum())
print("Missing/invalid dates after conversion:", invalid_date_count)
if invalid_date_count:
    df = df.dropna(subset=["Application Date"]).copy()
    print("Rows removed for invalid/missing dates:", invalid_date_count)

df["Phone"] = (df["Phone"].astype("string")
               .str.replace("-", "", regex=False)
               .str.replace(" ", "", regex=False))
df.info()

## Step 8 — Validate the cleaned data
Check missing values, duplicates, allowed domain labels, date type, and status labels. Placeholder emails are not genuine contact details.

In [ ]:
allowed_domains = {"Web Development", "Data Science", "Cyber Security",
                   "Artificial Intelligence", "App Development"}
print("Final shape:", df.shape)
print("\\nMissing values per column:")
display(df.isnull().sum())
print("Exact duplicate rows remaining:", int(df.duplicated().sum()))
print("Duplicate emails remaining:", int(df["Email"].duplicated().sum()))
print("Domain labels:", sorted(df["Domain Applied"].dropna().unique().tolist()))
print("Only approved domain labels:", set(df["Domain Applied"].dropna().unique()).issubset(allowed_domains))
print("Application Date dtype:", df["Application Date"].dtype)
print("Phone dtype:", df["Phone"].dtype)
print("Status labels:", sorted(df["Status"].dropna().unique().tolist()))

## Step 9 — Basic exploration after cleaning
These counts describe the dataset; no machine-learning model is built in this task.

In [ ]:
print("Applicants by domain:")
display(df["Domain Applied"].value_counts().rename_axis("Domain").to_frame("Applicants"))
print("Applicants by status:")
display(df["Status"].value_counts().rename_axis("Status").to_frame("Applicants"))
print("Applications by month:")
display(df["Application Date"].dt.to_period("M").value_counts().sort_index().rename_axis("Month").to_frame("Applications"))

## Step 10 — Data Quality Summary
The numbers below are calculated from the actual run, rather than copied from the task guide example.

In [ ]:
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"Starting dataset: {raw.shape[0]} rows, {raw.shape[1]} columns")
print(f"Final dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"- Rows dropped for missing Applicant Name: {missing_name_count}")
print(f"- Missing Email values replaced with labelled placeholders: {missing_email_count}")
print(f"- Missing University values filled: {missing_university_count}")
print(f"- Missing Status values filled: {missing_status_count}")
print(f"- Exact duplicate rows removed: {exact_duplicates_before}")
print(f"- Additional duplicate-email rows removed: {duplicate_email_count}")
print(f"- Missing/invalid dates found after parsing: {invalid_date_count}")
print(f"- Exact duplicates remaining: {int(df.duplicated().sum())}")
print(f"- Duplicate emails remaining: {int(df['Email'].duplicated().sum())}")
print(f"- Missing values remaining: {int(df.isna().sum().sum())}")
print("Text fields standardized; dates converted; phone numbers retained as text.")
print("Review placeholder emails before any real outreach.")

## Step 11 — Export the cleaned dataset
`index=False` prevents Pandas from adding an unnecessary row-number column.

In [ ]:
df.to_csv("applicants_cleaned.csv", index=False)
print("Saved cleaned dataset to applicants_cleaned.csv")
print("File size:", os.path.getsize("applicants_cleaned.csv"), "bytes")

## Final checklist
- [x] Explored data before cleaning
- [x] Handled missing values with explanations
- [x] Removed exact duplicates and repeated emails
- [x] Standardized text and domain labels
- [x] Converted dates and checked invalid dates
- [x] Cleaned phone formatting while preserving text type
- [x] Validated results and wrote a data quality summary
- [x] Exported the cleaned CSV

**Before submitting:** Run all cells from top to bottom, inspect unmapped domains, and upload both this notebook and `applicants_cleaned.csv` to your public GitHub repository.